In [1]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn nest_asyncio

# ── Cell 2: Imports + Config ──────────────────────────────────────────────────
import json, time, uuid, os, re, asyncio, subprocess, threading
from pathlib import Path
from typing import Optional
from datetime import datetime

import torch
import uvicorn
import nest_asyncio
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen2.5-14B-Instruct"   # 14B fits in 2xT4 (32GB total)
PORT          = 8002
SESSIONS_DIR  = Path("/kaggle/working/sessions")   # persisted to disk
SESSIONS_DIR.mkdir(exist_ok=True)

MAX_QUESTIONS     = 10    # max questions per interview
CONTEXT_WINDOW    = 6     # last N Q&A turns kept in live context
SUMMARY_THRESHOLD = 4     # summarize older turns beyond this

# ── Cell 3: Load Qwen on 2x T4 ───────────────────────────────────────────────
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB")

print(f"\nLoading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",          # splits across both T4s automatically
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / 1e9
used_vram = sum(
    torch.cuda.memory_allocated(i)
    for i in range(torch.cuda.device_count())
) / 1e9
print(f"Model loaded ✓  VRAM used: {used_vram:.1f} / {total_vram:.1f} GB")

# ── Cell 4: Inference ─────────────────────────────────────────────────────────
@torch.inference_mode()
def generate(messages: list, max_new_tokens: int = 512) -> str:
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ── Cell 5: Session Storage (disk-backed) ────────────────────────────────────
def session_path(session_id: str) -> Path:
    return SESSIONS_DIR / f"{session_id}.json"

def load_session(session_id: str) -> dict:
    p = session_path(session_id)
    if not p.exists():
        raise HTTPException(404, f"Session {session_id} not found")
    return json.loads(p.read_text())

def save_session(session: dict):
    session_path(session["session_id"]).write_text(
        json.dumps(session, indent=2)
    )

def new_session(cv_profile: dict, candidate_name: str) -> dict:
    session = {
        "session_id":      str(uuid.uuid4()),
        "candidate_name":  candidate_name,
        "cv_profile":      cv_profile,
        "created_at":      datetime.utcnow().isoformat(),
        "status":          "active",       # active | ended
        "question_count":  0,
        "history":         [],             # full Q&A list
        "rolling_summary": "",             # compressed old context
    }
    save_session(session)
    return session

# ── Cell 6: Context Builder ───────────────────────────────────────────────────
def build_cv_context(cv: dict) -> str:
    """Converts CV JSON into a compact context string for the system prompt."""
    p       = cv.get("personal", {})
    skills  = cv.get("skills", {})
    exp     = cv.get("experience", [])
    proj    = cv.get("projects", [])
    edu     = cv.get("education", [])

    lines = [
        f"Candidate: {p.get('name', 'Unknown')}",
        f"Seniority: {cv.get('seniority_level', 'unknown')} | "
        f"Total Experience: {cv.get('total_experience_years', 0)} years",
        f"Summary: {cv.get('summary_one_liner', '')}",
    ]

    if exp:
        lines.append("\nWork Experience:")
        for e in exp[:4]:  # top 4 roles
            lines.append(f"  • {e.get('role')} at {e.get('company')} ({e.get('duration', '')})")
            for h in e.get("highlights", [])[:2]:
                lines.append(f"    - {h}")

    if skills.get("technical"):
        lines.append(f"\nTechnical Skills: {', '.join(skills['technical'][:10])}")
    if skills.get("tools_and_frameworks"):
        lines.append(f"Tools/Frameworks: {', '.join(skills['tools_and_frameworks'][:8])}")

    if proj:
        lines.append("\nKey Projects:")
        for pr in proj[:3]:
            lines.append(f"  • {pr.get('name')}: {pr.get('description', '')[:100]}")

    if edu:
        lines.append("\nEducation:")
        for e in edu[:2]:
            lines.append(f"  • {e.get('degree')} in {e.get('field')} — {e.get('institution')} ({e.get('year', '')})")

    return "\n".join(lines)


def build_messages(session: dict, new_answer: Optional[str] = None) -> list:
    """
    Builds the message list for Qwen.
    Uses rolling summary for old context + recent turns in full.
    """
    cv_context = build_cv_context(session["cv_profile"])
    history    = session["history"]
    summary    = session["rolling_summary"]
    q_count    = session["question_count"]
    total_q    = MAX_QUESTIONS

    system = f"""You are a professional technical interviewer conducting a personalized job interview.

CANDIDATE PROFILE:
{cv_context}

INTERVIEW RULES:
- Ask ONE question at a time. Never ask multiple questions in one turn.
- Tailor questions to the candidate's actual experience and skills listed above.
- Start with a warm, specific opening question referencing their background.
- Progress: warm-up → technical depth → behavioral → situational → closing.
- Ask follow-up questions if an answer is vague or interesting.
- When question_count reaches {total_q}, wrap up professionally.
- Do NOT give feedback on answers during the interview.
- Respond with ONLY the next question, nothing else.

CURRENT PROGRESS: Question {q_count + 1} of {total_q}"""

    messages = [{"role": "system", "content": system}]

    # Add compressed summary of old turns if exists
    if summary:
        messages.append({
            "role": "user",
            "content": f"[Earlier interview summary]\n{summary}"
        })
        messages.append({
            "role": "assistant",
            "content": "Understood, continuing the interview."
        })

    # Add recent turns (last CONTEXT_WINDOW Q&As)
    recent = history[-CONTEXT_WINDOW:]
    for turn in recent:
        messages.append({"role": "assistant", "content": turn["question"]})
        messages.append({"role": "user",      "content": turn["answer"]})

    # Add new answer if provided
    if new_answer:
        messages.append({"role": "user", "content": new_answer})

    return messages


def maybe_summarize(session: dict):
    """
    If history exceeds SUMMARY_THRESHOLD, compress older turns into a rolling summary.
    Keeps recent turns in full for quality context.
    """
    history = session["history"]
    if len(history) <= SUMMARY_THRESHOLD:
        return

    # Turns to compress (all except last CONTEXT_WINDOW)
    to_compress = history[:-CONTEXT_WINDOW]
    if not to_compress:
        return

    # Already summarized up to this point?
    already_summarized = len(to_compress) <= session.get("_summarized_up_to", 0)
    if already_summarized:
        return

    turns_text = "\n".join(
        f"Q: {t['question']}\nA: {t['answer']}" for t in to_compress
    )
    summary_prompt = [
        {"role": "system", "content": "You are a concise summarizer. Summarize the key points from this interview exchange in 3-5 bullet points. Focus on what was revealed about the candidate's skills and experience."},
        {"role": "user",   "content": turns_text}
    ]

    try:
        summary = generate(summary_prompt, max_new_tokens=200)
        session["rolling_summary"] = summary
        session["_summarized_up_to"] = len(to_compress)
        print(f"  [Memory] Summarized {len(to_compress)} old turns")
    except Exception as e:
        print(f"  [Memory] Summary failed: {e}")

# ── Cell 7: Transcript Formatter ──────────────────────────────────────────────
def format_transcript(session: dict) -> str:
    """
    Formats session history into Q:/A: format
    compatible with the AI detector server.
    """
    lines = [
        f"# Interview Transcript",
        f"# Candidate: {session['candidate_name']}",
        f"# Date: {session['created_at']}",
        f"# Total Questions: {session['question_count']}",
        "="*60,
        ""
    ]
    for turn in session["history"]:
        lines.append(f"Q: {turn['question']}")
        lines.append(f"A: {turn['answer']}")
        lines.append("")
    return "\n".join(lines)

# ── Cell 8: FastAPI App ───────────────────────────────────────────────────────
app = FastAPI(title="Qwen Interview API")

class StartRequest(BaseModel):
    cv_profile:     dict          # JSON output from CV parser
    candidate_name: Optional[str] = "Candidate"

class AnswerRequest(BaseModel):
    session_id: str
    answer:     str

@app.get("/health")
def health():
    vram = {
        f"gpu_{i}": {
            "used_gb":  round(torch.cuda.memory_allocated(i)/1e9, 2),
            "total_gb": round(torch.cuda.get_device_properties(i).total_memory/1e9, 2),
        }
        for i in range(torch.cuda.device_count())
    }
    return {"status": "ok", "model": MODEL_ID, "vram": vram}


@app.post("/interview/start")
async def start_interview(req: StartRequest):
    """
    Start a new interview session.
    Pass the full JSON output from the CV parser.

    Returns: session_id + first question
    """
    session = new_session(req.cv_profile, req.candidate_name)
    print(f"→ New session {session['session_id']} for {req.candidate_name}")

    messages = build_messages(session)
    t0 = time.perf_counter()
    first_question = generate(messages, max_new_tokens=150)
    elapsed = round(time.perf_counter() - t0, 3)

    # Store first question (answer will come in next call)
    session["_pending_question"] = first_question
    session["question_count"]    = 1
    save_session(session)

    print(f"  Q1: {first_question[:80]}...  ({elapsed}s)")
    return JSONResponse({
        "session_id":      session["session_id"],
        "question_number": 1,
        "question":        first_question,
        "elapsed_s":       elapsed,
        "status":          "active"
    })


@app.post("/interview/answer")
async def submit_answer(req: AnswerRequest):
    """
    Submit candidate's answer, get next question.
    When max questions reached, returns status=ended with transcript.

    Returns: next question OR final transcript if done
    """
    session = load_session(req.session_id)

    if session["status"] == "ended":
        raise HTTPException(400, "Interview already ended. Use /interview/transcript to get results.")

    pending_q = session.get("_pending_question", "")
    if not pending_q:
        raise HTTPException(400, "No pending question. Call /interview/start first.")

    # Store completed Q&A turn
    session["history"].append({
        "question": pending_q,
        "answer":   req.answer.strip(),
        "turn":     session["question_count"],
    })

    # Compress old context if needed
    maybe_summarize(session)

    q_num = session["question_count"]

    # Check if interview is done
    if q_num >= MAX_QUESTIONS:
        session["status"]           = "ended"
        session["ended_at"]         = datetime.utcnow().isoformat()
        session["_pending_question"] = ""
        transcript = format_transcript(session)
        session["transcript"] = transcript
        save_session(session)

        print(f"  Session {req.session_id} ended after {q_num} questions")
        return JSONResponse({
            "session_id":      req.session_id,
            "status":          "ended",
            "message":         "Interview complete. Thank you!",
            "total_questions": q_num,
            "transcript":      transcript,
        })

    # Generate next question
    messages = build_messages(session)
    t0 = time.perf_counter()
    next_question = generate(messages, max_new_tokens=150)
    elapsed = round(time.perf_counter() - t0, 3)

    session["question_count"]    += 1
    session["_pending_question"]  = next_question
    save_session(session)

    print(f"  Q{session['question_count']}: {next_question[:80]}...  ({elapsed}s)")
    return JSONResponse({
        "session_id":      req.session_id,
        "question_number": session["question_count"],
        "question":        next_question,
        "elapsed_s":       elapsed,
        "status":          "active",
        "progress":        f"{session['question_count']}/{MAX_QUESTIONS}"
    })


@app.get("/interview/transcript/{session_id}")
async def get_transcript(session_id: str):
    """
    Get full transcript for a session.
    Format is compatible with AI detector /detect/transcript endpoint.
    """
    session = load_session(session_id)

    if session["status"] != "ended":
        raise HTTPException(400, "Interview not ended yet.")

    transcript = session.get("transcript") or format_transcript(session)
    return JSONResponse({
        "session_id":     session_id,
        "candidate_name": session["candidate_name"],
        "transcript":     transcript,
        "total_turns":    session["question_count"],
    })


@app.get("/interview/sessions")
async def list_sessions():
    """List all sessions (active + ended)."""
    sessions = []
    for f in SESSIONS_DIR.glob("*.json"):
        try:
            s = json.loads(f.read_text())
            sessions.append({
                "session_id":      s["session_id"],
                "candidate_name":  s["candidate_name"],
                "status":          s["status"],
                "question_count":  s["question_count"],
                "created_at":      s["created_at"],
            })
        except: pass
    return JSONResponse({"sessions": sessions})


# ── Cell 9: Start localhost.run tunnel ───────────────────────────────────────
def start_tunnel(port: int):
    proc = subprocess.Popen(
        ["ssh", "-o", "StrictHostKeyChecking=no",
         "-R", f"80:localhost:{port}",
         "nokey@localhost.run"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        line = line.decode("utf-8", errors="ignore").strip()
        match = re.search(r"https://[a-zA-Z0-9\-]+\.lhr\.life", line)
        if match:
            url = match.group(0)
            print("\n" + "="*60)
            print(f"  Qwen Interview API  ->  {url}")
            print(f"  Docs               ->  {url}/docs")
            print(f"  Health             ->  {url}/health")
            print("="*60 + "\n")
            break

threading.Thread(target=start_tunnel, args=(PORT,), daemon=True).start()
time.sleep(5)

# ── Cell 10: Start server (blocks) ───────────────────────────────────────────
nest_asyncio.apply()
config = uvicorn.Config(app, host="0.0.0.0", port=PORT)
server = uvicorn.Server(config)
asyncio.get_event_loop().run_until_complete(server.serve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.5 MB/s eta 0:00:00:00:0100:01
GPUs available: 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB

Loading Qwen/Qwen2.5-14B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded ✓  VRAM used: 28.0 / 31.3 GB

  Qwen Interview API  ->  https://ad1aa3235c0919.lhr.life
  Docs               ->  https://ad1aa3235c0919.lhr.life/docs
  Health             ->  https://ad1aa3235c0919.lhr.life/health



INFO:     Started server process [55]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [55]


KeyboardInterrupt: 